# 01 — Dataset exploration: CyberLab honeynet

**Goal:** understand the raw data before analysing any of it.

1. What the files are (format, compression) and how to load them
2. The columns / fields, and what each one means
3. How many events of each `eventid` type there are

Deliberately limited to understanding. No findings, no conclusions.

---

## Dataset

| | |
|---|---|
| **Name** | CyberLab honeynet dataset |
| **Source** | [Zenodo record 3687527](https://zenodo.org/records/3687527) |
| **Authors** | Sedlar, Kren, Stefanič Južnič, Volk — University of Ljubljana, FEE |
| **Published** | 2020-02-29 |
| **Honeypot** | Cowrie, ~50 nodes at EU and US universities and companies |
| **Full range** | May 2019 – Feb 2020 (~10 GB, ~290 daily files) |
| **Analysed here** | **3 days only: 2020-02-24, 2020-02-28, 2020-02-29** |

### Scope caveats — these constrain every conclusion

**1. Three days is not the full range.** The dataset spans ~10 months; this
sample covers three days at its very end. No seasonality or trend claim is
supported.

**2. Mixed sensor configurations.** These days are *not* a single experiment
config. `sensor` splits them as follows:

| `sensor` | 2020-02-24 | 2020-02-28 | 2020-02-29 |
|---|---:|---:|---:|
| `ubuntu_basic_pool` | 89.05% | 81.06% | 80.36% |
| `prod-ubuntu-ssh-k8s-local-1` | 2.52% | 4.63% | 5.50% |
| `prod-ubuntu-ssh-k8s-local-2` | 2.43% | 4.43% | 3.75% |
| `prod-ubuntu-ssh-k8s-local-3` | 4.05% | 4.83% | 4.53% |
| `prod-ubuntu-ssh-k8s-local-4` | 1.95% | 5.04% | 5.86% |

`ubuntu_basic_pool` is the high-interaction mode enabled when the fleet moved
from Cowrie 1.6.0 to 2.0.2 on 2019-11-08, backed by a pool of real Linux
instances. The four `prod-ubuntu-ssh-k8s-local-*` nodes are a different
configuration. Because honeypot mode changes what an attacker can observe and
do, a difference in `eventid` mix between sensors is **not** automatically a
difference in attacker behaviour.

**3. No pre-upgrade data.** All three days fall after 2019-11-08.

**4. Node count varied.** "~50 nodes" is the peak, not a constant. Five distinct
`sensor` values appear in these three files alone.

### Safety rules for this project

- **Never** open, execute, or analyse the `data` field — it can hold the contents
  of attacker-uploaded files. The loader below drops it automatically.
- Raw files live in `data/raw/`, which is git-ignored and never committed.
- Anything published goes to `data/processed/`.
- Attacker IPs are **pseudonymized** by the dataset authors as SHA-256 hashes.
  That is *not* anonymisation — see the note in the data dictionary below.
- `username` / `password` are **plaintext** attacker-typed credentials. They are
  sensitive and must be redacted before any public output.

## 0. Setup

In [ ]:
import gzip
import json
import pathlib
import re
import time

import pandas as pd

pd.set_option("display.max_colwidth", 90)


def find_repo_root(start=pathlib.Path.cwd()):
    '''Walk upward until we find the directory containing data/raw/.

    Makes the notebook runnable from the repo root or from analysis/,
    instead of silently producing 'no such file' errors.
    '''
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ - run this notebook from inside the repository."
    )


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "raw"

files = sorted(DATA_DIR.glob("*.json.gz"))
print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"Files     : {len(files)}\n")

for f in files:
    print(f"  {f.name:<32} {f.stat().st_size / 1e6:>7.1f} MB")

### Unwrapping a line

The whole file is **one JSON array**, so an individual line is only a *fragment*
of that array. It cannot be handed to `json.loads` until the array punctuation is
removed:

- the **first** line starts with `[`
- every line **except the last** ends with `,`
- the **last** line ends with `]` and has **no trailing newline**

That last point is worth remembering: `wc -l` counts newlines, so it reports one
fewer than the true number of lines. `wc -l` says 47,029 for the first file;
there are actually 47,030 sessions in it.

In [ ]:
def unwrap_line(line):
    '''Strip JSON-array punctuation from one line of a CyberLab daily file.

    Returns the inner JSON object text, or '' for a blank line.
    '''
    s = line.strip()
    if s.startswith("["):      # only the first line
        s = s[1:]
    s = s.rstrip(",")           # every line except the last
    if s.endswith("]"):        # only the last line
        s = s[:-1]
    return s.strip()


print(repr(unwrap_line('[{"abc": [1]},')))
print(repr(unwrap_line('{"def": [2]},')))
print(repr(unwrap_line('  ')))

## 1. What is inside these files?

Three facts to establish: **compression**, **structure**, and **nesting**.

In [ ]:
SAMPLE_FILE = files[0]
print(f"Inspecting: {SAMPLE_FILE.name}\n")

with gzip.open(SAMPLE_FILE, "rt", encoding="utf-8") as fh:
    raw_first = fh.readline()

HASH_RE = re.compile(r"\b[0-9a-f]{64}\b")
safe_preview = HASH_RE.sub("<sha256>", raw_first)

print("--- raw first 300 characters (pseudonyms replaced) ---")
print(safe_preview[:300], "...\n")

record = json.loads(unwrap_line(raw_first))

print("--- what that line actually contained ---")
print(f"type(parsed line)      : {type(record).__name__}")
print(f"number of session keys : {len(record)}")
print(f"keys                   : {list(record.keys())[:3]} ...")

In [ ]:
first_session_id = list(record.keys())[0]
events = record[first_session_id]

print(f"Session {first_session_id!r} holds {len(events)} events:")
for e in events:
    print(f"    {e['eventid']:<28} {e['timestamp']}")

### The key structural fact

Each file is a **single JSON array**, laid out one attack session per line:

```
[
  {"<session_id>": [ {event}, {event}, ... ]},
  {"<session_id>": [ {event}, {event}, ... ]},
  ...
]
```

Cowrie's own log format is one event per line. The CyberLab authors **regrouped**
the raw log into sessions before publishing — friendlier to read by hand, but
awkward for every off-the-shelf pandas reader.

It is valid JSON as a whole. It is **not** JSON Lines.

### What each candidate reader actually does

| Attempt | Outcome |
|---|---|
| `pd.read_json(path)` | **Succeeds** — the file is one valid JSON document. But it must parse the *entire* array into memory as nested Python objects before it builds anything. 1.4 GB for three days; impossible for the full ~10 GB set. Correct, but it defeats streaming. |
| `pd.read_json(path, lines=True)` | **Fails.** `lines=True` expects one complete JSON value per line. Here each line is a *fragment* of an array, so the parser hits an unterminated array or a trailing comma. |
| `pd.read_csv(path, compression="gzip")` | **Fails** — the content is JSON, not delimited text. |

So we stream it ourselves. That costs some speed and buys control — including the
ability to drop the attacker-payload field as we go.

## 2. Loading

The loader unwraps each line, then walks the nested structure and yields **one
flat event dict at a time**.

Two deliberate details:

- **`drop_payload=True`** removes the `data` field, which can contain the contents
  of an attacker-uploaded file. We never read attacker payloads — so the loader
  refuses to hand them to us, rather than relying on discipline later.
- **`limit_sessions`** lets you iterate fast on a slice before committing to a
  full file.

In [ ]:
def iter_events(path, drop_payload=True, limit_sessions=None):
    '''Yield one flat event dict at a time from a CyberLab daily file.

    The file is one JSON array with one session per line, so this unwraps
    each line and walks the nested structure manually, without ever holding
    more than one line in memory.

    Parameters
    ----------
    drop_payload : bool
        Drop the `data` field. It can contain attacker-uploaded file
        contents, which this project never inspects. (Safety rule.)
    limit_sessions : int or None
        Stop after this many sessions (i.e. after this many lines).
        Sessions per file vary, so this is a sample, not a fraction.
    '''
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for n, line in enumerate(fh):
            if limit_sessions is not None and n >= limit_sessions:
                return
            body = unwrap_line(line)
            if not body:
                continue
            for events in json.loads(body).values():
                for ev in events:
                    if drop_payload:
                        ev.pop("data", None)
                    yield ev

### Quick sample first — 2,000 sessions

In [ ]:
quick = pd.DataFrame.from_records(iter_events(SAMPLE_FILE, limit_sessions=2000))
print(f"{len(quick):,} events from the first 2,000 sessions\n")
quick["eventid"].value_counts().to_frame("events")

### Now one full day

In [ ]:
t0 = time.perf_counter()
df = pd.DataFrame.from_records(iter_events(SAMPLE_FILE))
elapsed = time.perf_counter() - t0

print(f"File      : {SAMPLE_FILE.name}")
print(f"Events    : {len(df):,}")
print(f"Columns   : {df.shape[1]}")
print(f"Load time : {elapsed:.1f}s")
print(f"Memory    : {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB\n")

df.shape

## 3. The columns, and what each one means

First the mechanical view — dtype, how often each field is populated, and a
truncated example value.

In [ ]:
# Columns whose values are withheld from the example column below: attacker
# and honeypot pseudonyms, credentials, and free-text that restates them.
SENSITIVE_COLUMNS = {
    "src_ip_identifier", "dst_host_identifier", "message",
    "username", "password",
}


def brief(value, width=55):
    text = repr(value)
    return text if len(text) <= width else text[:width] + "..."


overview = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "example": [
            "(withheld)" if c in SENSITIVE_COLUMNS
            else (brief(df[c].dropna().iloc[0]) if df[c].notna().any() else None)
            for c in df.columns
        ],
    }
)
overview

Most columns are sparse. Cowrie emits a single wide schema and fills in only the
fields relevant to a given event — so `password` is populated on login events and
`null` everywhere else. A low non-null count here is expected, not a defect.

In [ ]:
DATA_DICTIONARY = [
    ("session_id", "Identity", "Groups every event belonging to one SSH session.", "Also the JSON key used to group events in the file."),
    ("src_ip_identifier", "Identity", "Attacker IP, stored as a SHA-256 hash.", "Pseudonymised, NOT anonymised: the IPv4 space is only 2^32, so the hash can be brute-forced by enumerating it. Treat as personal data (GDPR)."),
    ("dst_ip_identifier", "Identity", "Destination IP, stored as a SHA-256 hash.", "Frequently null."),
    ("dst_host_identifier", "Identity", "Pseudonymized name of the honeypot node.", "Tells you which node was attacked."),
    ("eventid", "Event", "Event type, e.g. cowrie.login.success.", "The main categorical dimension of the dataset."),
    ("timestamp", "Event", "Event time, UTC, ISO-8601 with microseconds.", "Parsed as string until you convert it."),
    ("message", "Event", "Cowrie's human-readable log line.", "Free text. Can restate passwords, so treat as sensitive."),
    ("protocol", "Network", "ssh or telnet.", "This sample is ssh only."),
    ("src_port", "Network", "Attacker's ephemeral TCP port.", "Not comparable across sessions."),
    ("sensor", "Sensor", "Honeypot node group / experiment config.", "See the scope table in cell 1 - five values appear here, and they are NOT the same experiment config."),
    ("arch", "Sensor", "CPU/OS architecture emulated by Cowrie.", "Sparse."),
    ("duration", "Session", "Session length in seconds.", "Only populated on cowrie.session.closed."),
    ("ssh_client_version", "Client fingerprint", "Attacker's SSH banner string.", "Stored as a Python bytes repr, e.g. b'SSH-2.0-Go'. Needs cleaning."),
    ("hassh", "Client fingerprint", "HASSH: hash of the negotiated SSH algorithms.", "Strong malware-family indicator."),
    ("hasshAlgorithms", "Client fingerprint", "Full algorithm-negotiation string.", "One string, not a list."),
    ("kexAlgs", "Client fingerprint", "Key-exchange algorithms offered.", "List."),
    ("keyAlgs", "Client fingerprint", "Public-key algorithms offered.", "List."),
    ("encCS", "Client fingerprint", "Encryption algorithms offered.", "List."),
    ("macCS", "Client fingerprint", "MAC algorithms offered.", "List."),
    ("compCS", "Client fingerprint", "Compression algorithms offered.", "List."),
    ("langCS", "Client fingerprint", "Language algorithms offered.", "List, usually a single empty string."),
    ("fingerprint", "Client fingerprint", "Host key fingerprint presented.", "Populated on cowrie.client.fingerprint events (~0.03-0.08% of volume)."),
    ("key", "Client fingerprint", "Public key offered by the client.", "Rarely populated."),
    ("username", "Auth", "Login username attempted.", "Plaintext, attacker-typed. Sensitive."),
    ("password", "Auth", "Password attempted.", "Plaintext, attacker-typed. Sensitive - redact before publishing."),
    ("type", "File transfer", "Kind of transferred object.", "e.g. a downloaded file."),
    ("filename", "File transfer", "Filename as given by the attacker.", "Attacker-controlled string."),
    ("outfile", "File transfer", "Where Cowrie wrote the file on the honeypot.", "Honeypot-side path."),
    ("destfile", "File transfer", "Path on the emulated filesystem.", ""),
    ("shasum", "File transfer", "Checksum of the transferred file.", "Hash only. We never open the file."),
    ("url", "File transfer", "Download URL.", "Strong indicator of compromise."),
    ("size", "File transfer", "File size in bytes.", ""),
    ("ttylog", "File transfer", "Reference to the recorded terminal session.", "Points at a capture file. Do not open."),
    ("duplicate", "File transfer", "Marks a repeat of an already-seen file.", "Useful for de-duplication."),
    ("data", "File transfer", "Raw contents of the transferred file.", "NEVER inspect. Dropped by the loader by default."),
    ("geolocation_data", "Geolocation", "Nested dict: country, city, lat/lon, timezone.", "Often an empty {} - handle missing geo, not just wrong geo."),
]

dictionary = pd.DataFrame(DATA_DICTIONARY, columns=["field", "category", "meaning", "note"])
print(f"{len(dictionary)} fields documented\n")
dictionary

Does the dictionary actually cover the data? Checking both directions:

In [ ]:
in_data = set(df.columns)
in_doc = set(dictionary["field"])

print(f"columns in data but NOT documented : {sorted(in_data - in_doc) or 'none'}")
print(f"documented but NOT in data         : {sorted(in_doc - in_data) or 'none'}")
print(f"documented and present            : {len(in_data & in_doc)}")

Note that Zenodo's published field table omits about 17 of these fields (`hassh`,
`arch`, `url`, `shasum`, `size`, `filename`, `ttylog`, `data`, and others). The
official documentation is incomplete relative to the actual data.

### Two traps in the "File transfer" category

**1. `shasum` here is not a malware hash.** The rows with a non-null `shasum` are
all `cowrie.log.closed` events - session-log lifecycle, not downloads. Every one
of them carries the *same* checksum, `e3b0c44298fc...b855`, which is the SHA-256
of the **empty string**, while `size` differs per event (5,154 / 37 / 34 / 113
bytes observed). So `shasum` is functioning as a log identifier, not a content
hash.

The consequence is concrete: grouping by `shasum` and counting would suggest
"one file was downloaded 3,517 times". That reading is wrong, and it would be an
embarrassing error in a published report. **Never use `shasum` as a malware
indicator in this dataset.**

**2. There are no file downloads in this sample at all.** `url`, `filename`,
`outfile` and `destfile` are 100% null on all three days. Any analysis of
download URLs or IOCs will return an empty result until more days are added.

### Days are partitioned by session, not by event timestamp

`2020-02-24` holds 6 events timestamped 02-25, and `2020-02-28` holds 18
timestamped 02-29 - sessions that *started* before midnight.

This does **not** double-count: session IDs are disjoint across adjacent files
(67,813 + 71,172 = 138,985 distinct, 0 shared). But `timestamp` spans a slightly
wider range than the filename suggests, which matters for any hourly or daily
time series.

## 4. How many events of each type?

Ten distinct `eventid` values appear in this file. The shape is worth reading
before drawing conclusions from it: the five most common events
(`login.failed`, `session.connect`, `session.closed`, `client.version`,
`client.kex`) together account for roughly 95% of all events and mostly describe
the *protocol handshake* and *failed authentication*, not attacker activity.

The genuinely interesting events are the small remainder: `command.input`,
`login.success` and `direct-tcpip.request`.

Note that `file.download` does **not** appear in this data at all — `url`,
`filename`, `outfile` and `destfile` are 100% null. The `shasum` values that do
appear belong to `log.closed` events and identify session-log files, not
downloads.

### `cowrie.direct-tcpip.request` — relay / pivoting attempts

This event fires when an SSH client asks the honeypot to open a TCP connection
to some other host on its behalf. That is the `direct-tcpip` channel type in
SSH: the client is trying to **use the honeypot as a relay**, so traffic to
the real target appears to come from the honeypot rather than from the
attacker. In MITRE ATT&CK this is **T1090 — Proxy**.

It deserves a section in the report despite being only 0.76% of events,
because it is direct evidence of pivoting intent rather than brute-force
background noise, and it is the one event type here that implies an
*infrastructure* objective rather than a credential objective.

The `message` field carries the request:

```
direct-tcp connection request to dab4fb8317d15c50...:80 from 127.0.0.1:7557
```

Three things to know before analysing it:

- **The destination is pseudonymized too** — a SHA-256 hash, same treatment as
  `src_ip_identifier`. Distinct targets can be counted and ranked, but they
  cannot be geolocated or attributed.
- **`127.0.0.1` is the honeypot's own loopback view**, not the attacker. The
  apparent source there is the emulated host and is useless as an indicator.
- **Targets are heavily concentrated.** Across all three days there are 6,027
  requests to only **247 distinct targets**, and the top 5 account for **49%**
  of all requests; the single most-targeted host takes 21%. Ports are almost
  entirely **80** (91%), with **25** (SMTP) a distant second at 8.7%.

The questions worth asking in the report: how many distinct targets per
session, how concentrated the target set is, and whether the sessions doing
heavy relaying differ from the bulk population in duration, in sensor, or in
whether they reached `login.success` first.

In [ ]:
counts = df["eventid"].value_counts()
result = counts.to_frame("events")
result["pct_of_all"] = (result["events"] / len(df) * 100).round(2)
result

Sanity checks on the loaded frame — is it internally consistent?

In [ ]:
print("Sensors (node groups) in this file:")
print(df["sensor"].value_counts().to_string())
print(f"\nDistinct sessions          : {df['session_id'].nunique():,}")
print(f"Distinct attacker IPs      : {df['src_ip_identifier'].nunique():,} (hashed)")
print(f"Total events               : {len(df):,}")
print(f"Events per session (mean)  : {len(df) / df['session_id'].nunique():.2f}")
print(f"\nTime range                 : {df['timestamp'].min()}  ->  {df['timestamp'].max()}")

## What we established

1. **Format** — gzip-compressed **single JSON array**, one attack session per line.
   Valid JSON overall, but not JSON Lines.
2. **Loading** — `pd.read_json` works but loads everything into memory first;
   `lines=True` fails. Streaming with `unwrap_line` + `iter_events` is what makes
   the full dataset tractable.
3. **Schema** — 36 fields in the raw schema; **35 reach the DataFrame**, because
   the loader drops `data` by design, and the coverage check above confirms that
   is the only difference. Fields span identity, event, network,
   client-fingerprinting, auth, file-transfer and geolocation categories. Most
   are sparse by design.
4. **Volume** — ~250k–380k events per day, from ~47k–71k sessions.
5. **Sensors** — five distinct configurations, not one experiment.

## Deliberately not done yet

No findings, no rankings, no charts. Before drawing conclusions:

- **Load all three days.** Everything above is one file, and session counts per
  day differ, so nothing here generalises to the sample.
- **Parse `timestamp` to real datetimes.** It is a string right now.
- **Clean `ssh_client_version`.** The leading `b` is a Python bytes-repr artefact
  that will corrupt any group-by or chart built on it.
- **Handle empty `geolocation_data`.** An empty dict is common and will silently
  drop rows from a naive geo filter.
- **Control for `sensor`.** Five configurations share these days; a raw aggregate
  across all of them may be comparing honeypot modes rather than attackers.
- **Reduce columns before scaling.** One day already holds ~254k rows. The full
  10 GB set needs a chunked strategy and a deliberate column selection.
- **Redact credentials.** `password` is plaintext and must never reach a public
  output.